# Day 6 — End-to-End ML Pipeline & Business Insights

This notebook integrates results from:
- Customer & restaurant segmentation (Day 2)
- Sentiment analysis (Day 3)
- Delivery time prediction (Day 4)
- Demand forecasting (Day 5)

It does not retrain any models — it loads saved outputs and focuses
on business interpretation.

In [5]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Loading Saved Outputs

- Customer segmentation
- Restaurant segmentation
- Reviews with sentiment labels

In [6]:
# Load customer segmentation results

customer_segments = pd.read_csv(
    '../data/processed/customer_segmentation.csv'
)

# Load restaurant segmentation results

restaurant_segments = pd.read_csv(
    '../data/processed/restaurant_segmentation.csv'
)

# Load reviews with sentiment labels

reviews_sentiment = pd.read_csv(
    '../data/processed/reviews_cleaned.csv'
)

# Confirm shapes

print('Customer segments:', customer_segments.shape)
print('Restaurant segments:', restaurant_segments.shape)
print('Reviews with sentiment:', reviews_sentiment.shape)

Customer segments: (1500, 11)
Restaurant segments: (200, 7)
Reviews with sentiment: (7500, 16)


## Customer Value Analysis

Comparing customer segments on ordering behavior and spending.

In [7]:
# Average metrics per customer segment

customer_segment_summary = (
    customer_segments
    .groupby('customer_segment')[
        [
            'total_orders',
            'average_order_value',
            'total_spending',
            'ordering_frequency',
            'weekend_orders',
            'late_night_orders'
        ]
    ]
    .mean()
    .round(2)
)

# Add segment sizes

customer_segment_summary['num_customers'] = (
    customer_segments['customer_segment']
    .value_counts()
)

customer_segment_summary

,total_orders,average_order_value,total_spending,ordering_frequency,weekend_orders,late_night_orders,num_customers
customer_segment,,,,,,,
Frequent High-Value Customers,9.08,855.24,7538.22,2.14,2.83,1.33,612
Occasional Moderate-Value Customers,5.01,661.01,3194.71,1.48,1.28,0.54,888


### Which customer segments are most valuable?

- Frequent High-Value Customers: 612 customers (41%)
  - 9.08 avg orders, 855 avg order value, 7,538 total spending
  - More weekend and late-night orders

- Occasional Moderate-Value Customers: 888 customers (59%)
  - 5.01 avg orders, 661 avg order value, 3,195 total spending

- Frequent High-Value Customers order ~2x more and spend >2x more in
  total, despite being the smaller group.
- Priority: retention (loyalty programs) for this segment; engagement
  campaigns to increase order frequency in the larger Occasional
  segment.

In [8]:
# Display restaurant segmentation columns

restaurant_segments.columns.tolist()

['restaurant_id',
 'total_orders',
 'average_order_value',
 'total_revenue',
 'average_delivery_time',
 'restaurant_rating',
 'cluster']

## Restaurant Performance Analysis

Comparing restaurant segments on revenue, order volume, delivery
performance, and rating.

In [9]:
# Map cluster numbers to readable labels

restaurant_segments['restaurant_segment'] = restaurant_segments['cluster'].map({
    0: 'High-Value Restaurants',
    1: 'Standard-Value Restaurants'
})

# Average metrics per restaurant segment

restaurant_segment_summary = (
    restaurant_segments
    .groupby('restaurant_segment')[
        [
            'total_orders',
            'average_order_value',
            'total_revenue',
            'average_delivery_time',
            'restaurant_rating'
        ]
    ]
    .mean()
    .round(2)
)

# Add segment sizes

restaurant_segment_summary['num_restaurants'] = (
    restaurant_segments['restaurant_segment']
    .value_counts()
)

restaurant_segment_summary

,total_orders,average_order_value,total_revenue,average_delivery_time,restaurant_rating,num_restaurants
restaurant_segment,,,,,,
High-Value Restaurants,47.37,2146.60,99818.97,51.35,3.96,30
Standard-Value Restaurants,50.46,521.55,26210.17,50.07,4.12,170


### Which restaurant segments require intervention?

- High-Value Restaurants: 30 restaurants (15%)
  - Avg order value 2,147, total revenue 99,819
  - Similar order count and delivery time to Standard-Value group
  - Rating slightly lower (3.96 vs 4.12)

- Standard-Value Restaurants: 170 restaurants (85%)
  - Avg order value 522, total revenue 26,210

- Delivery time and rating are similar across both segments —
  segmentation is driven by order value, not service quality.
- High-Value Restaurants' slightly lower rating is worth monitoring
  given their outsized revenue impact per restaurant.
- Standard-Value Restaurants don't show delivery or rating problems,
  so intervention isn't clearly indicated by this data alone — worth
  checking against sentiment analysis next.

In [10]:
# Total revenue contribution by restaurant segment

revenue_by_segment = (
    restaurant_segments
    .groupby('restaurant_segment')['total_revenue']
    .sum()
)

revenue_share = (
    revenue_by_segment / revenue_by_segment.sum() * 100
).round(1)

print(revenue_by_segment)
print('\nRevenue share (%):')
print(revenue_share)

restaurant_segment
High-Value Restaurants        2994569.03
Standard-Value Restaurants    4455729.30
Name: total_revenue, dtype: float64

Revenue share (%):
restaurant_segment
High-Value Restaurants        40.2
Standard-Value Restaurants    59.8
Name: total_revenue, dtype: float64


### How much revenue does each restaurant segment contribute?

- High-Value Restaurants (15% of restaurants): 40.2% of total revenue
  (2,994,569)
- Standard-Value Restaurants (85% of restaurants): 59.8% of total
  revenue (4,455,729)
- High-Value Restaurants punch well above their numbers — 1 in 7
  restaurants drives 2 in 5 revenue dollars. Losing or underserving
  even a few of these restaurants has outsized business impact
  compared to losing a Standard-Value restaurant.

In [11]:
# Display sentiment distribution overall

reviews_sentiment['sentiment'].value_counts()

sentiment
Positive    6019
Neutral     1395
Negative      86
Name: count, dtype: int64

### What is the overall sentiment distribution?

- Positive: 6,019 (80.3%)
- Neutral: 1,395 (18.6%)
- Negative: 86 (1.1%)
- Negative reviews are rare, but still worth examining closely since
  each one represents a concrete failure point.

In [12]:
# Merge review sentiment with restaurant segment

reviews_with_segment = reviews_sentiment.merge(
    restaurant_segments[['restaurant_id', 'restaurant_segment']],
    on='restaurant_id',
    how='left'
)

# Calculate sentiment share by restaurant segment

sentiment_by_segment = (
    reviews_with_segment
    .groupby('restaurant_segment')['sentiment']
    .value_counts(normalize=True)
    .unstack()
    .round(3) * 100
)

sentiment_by_segment

sentiment,Negative,Neutral,Positive
restaurant_segment,,,
High-Value Restaurants,0.9,18.8,80.3
Standard-Value Restaurants,1.2,18.6,80.3


### Does restaurant segment relate to customer sentiment?

- High-Value Restaurants: 0.9% negative, 18.8% neutral, 80.3% positive
- Standard-Value Restaurants: 1.2% negative, 18.6% neutral, 80.3% positive
- Sentiment distribution is nearly identical across both segments.
- Confirms earlier finding: restaurant segment (order value) is not
  linked to service quality or customer satisfaction. Intervention
  should not be targeted by segment — it should be targeted at
  individual restaurants with poor reviews, regardless of segment.

In [13]:
# Isolate negative reviews

negative_reviews = reviews_sentiment[
    reviews_sentiment['sentiment'] == 'Negative'
]

negative_reviews[['review_id', 'restaurant_id', 'review_rating', 'clean_text']].head(10)

,review_id,restaurant_id,review_rating,clean_text
12,REV_000013,REST_0144,2,order could much better
17,REV_000018,REST_0135,2,order could much better food expectation order...
73,REV_000074,REST_0150,2,disappointed order meal taste fresh delivery d...
74,REV_000075,REST_0157,2,order could much better food expectation deliv...
156,REV_000157,REST_0054,2,meal taste fresh order took much longer expected
230,REV_000231,REST_0066,2,satisfied order
371,REV_000372,REST_0119,2,food quality need improvement order took much ...
611,REV_000612,REST_0032,2,overall experience disappointing
819,REV_000820,REST_0102,2,satisfied order
900,REV_000901,REST_0020,2,food arrived cold delivery experience poor


In [14]:
# Count word frequency across all negative reviews

from collections import Counter

all_words = ' '.join(negative_reviews['clean_text'].dropna()).split()

word_counts = Counter(all_words)

# Display the 15 most common words

pd.Series(word_counts).sort_values(ascending=False).head(15)

order            45
food             44
delivery         41
experience       38
fresh            36
much             26
taste            19
poor             19
expected         18
meal             17
arrived          17
disappointing    17
extremely        15
slow             15
happy            12
dtype: int64

### What are the biggest causes of negative customer experiences?

Most frequent words in negative reviews (n=86):
- delivery (41), expected (18), slow (15) → delivery delays
- food (44), fresh (36), taste (19), arrived (17) → food quality on arrival
- experience (38), poor (19), disappointing (17) → general dissatisfaction

Two dominant themes:
1. Delivery took longer than expected
2. Food quality/freshness issues on arrival (likely tied to delivery
   time — food arriving cold/not fresh)

Both themes point to delivery time as a probable root cause,
connecting sentiment analysis directly to the delivery prediction
findings from Day 4.

## Delivery Time Analysis

In [15]:
# Feature importance results from Day 4 Random Forest model

delivery_feature_importance = pd.Series({
    'delivery_distance_km': 0.775954,
    'preparation_time_min': 0.126851,
    'traffic_condition_Severe': 0.032743,
    'traffic_condition_Low': 0.023686,
    'traffic_condition_Medium': 0.020067,
    'tip_amount': 0.004729,
    'order_amount': 0.004554,
    'order_hour': 0.003262,
    'restaurant_rating': 0.003249,
    'day_of_week': 0.002044,
    'items_count': 0.001660
}).sort_values(ascending=False)

delivery_feature_importance

delivery_distance_km        0.775954
preparation_time_min        0.126851
traffic_condition_Severe    0.032743
traffic_condition_Low       0.023686
traffic_condition_Medium    0.020067
tip_amount                  0.004729
order_amount                0.004554
order_hour                  0.003262
restaurant_rating           0.003249
day_of_week                 0.002044
items_count                 0.001660
dtype: float64

### What factors drive delivery delays?

From the Day 4 Random Forest model (test R² = 0.884):

- Delivery distance: 77.6% of predictive importance
- Preparation time: 12.7%
- Traffic condition (combined): 7.6%
- All other factors (tip, order amount, time, rating, etc.): <2% combined

Delivery distance and preparation time together account for over 90%
of what drives delivery time. Traffic adds a smaller but real effect.
Order and restaurant characteristics have minimal impact.

This directly supports the negative-review themes above: delivery
delays are structurally tied to distance and kitchen prep time, not
random or restaurant-quality issues.

## Demand Forecasting Analysis

In [16]:
# Model comparison results from Day 5

demand_model_comparison = pd.DataFrame({
    'Model': ['ARIMA', 'Prophet'],
    'MAE': [2.02, 1.04],
    'RMSE': [2.47, 1.48]
})

demand_model_comparison

,Model,MAE,RMSE
0,ARIMA,2.02,2.47
1,Prophet,1.04,1.48


### Which forecasting model performs better?

- Prophet: MAE 1.04, RMSE 1.48
- ARIMA: MAE 2.02, RMSE 2.47
- Prophet has roughly half the error of ARIMA. ARIMA converged to a
  flat average and missed the daily demand cycle; Prophet captured it.
- Prophet is the recommended model for operational forecasting.

In [17]:
# Peak demand hours identified in Day 5

peak_hours_summary = pd.Series({
    'Midday peak (11:00-14:00)': 'avg ~4.3 orders/hour',
    'Evening peak (18:00-22:00)': 'avg ~5.2 orders/hour',
    'Overnight low (0:00-6:00)': 'avg ~0.2 orders/hour'
})

peak_hours_summary

Midday peak (11:00-14:00)     avg ~4.3 orders/hour
Evening peak (18:00-22:00)    avg ~5.2 orders/hour
Overnight low (0:00-6:00)     avg ~0.2 orders/hour
dtype: str

### When should additional delivery capacity be deployed?

- Evening peak (18:00-22:00): ~5.2 orders/hour — highest demand window
- Midday peak (11:00-14:00): ~4.3 orders/hour — second peak
- Overnight (0:00-6:00): ~0.2 orders/hour — minimal demand
- Day-of-week demand is nearly flat (Day 5 finding) — no strong
  weekend vs weekday difference.
- Additional delivery capacity should be scheduled around the two
  daily peak windows (especially evening), not by day of week.

### Which time periods are likely to see demand spikes?

- The Day 5 Prophet forecast shows the same two-peak daily pattern
  repeating consistently across all 7 forecasted days.
- No day-of-week variation was found, so spikes are driven by time of
  day, not particular days.
- Note: the dataset does not include location/zone information, so
  demand spikes can only be identified by time period here, not by
  location.

## Model Limitations

## Model Limitations

| Component | Limitation |
|---|---|
| Customer/Restaurant Segmentation | Moderate silhouette scores (0.248 customers, 0.421 restaurants) — clusters are usable but not sharply separated |
| Sentiment Analysis | Severe class imbalance (86 negative vs 6,019 positive reviews) — negative-class performance may not generalize well |
| Delivery Time Prediction | Relies heavily on distance and prep time (90%+ combined importance) — other features add little, limiting nuance |
| Demand Forecasting (ARIMA) | Missed the daily seasonal cycle entirely — forecast converged to a flat average |
| Demand Forecasting (Prophet) | Uncertainty interval briefly goes negative during low-demand hours, not meaningful for order counts |

## Future Improvement Opportunities

| Area | Opportunity |
|---|---|
| Segmentation | Test additional features (e.g. cuisine type, customer tenure) to improve cluster separation |
| Sentiment Analysis | Collect more negative-labeled reviews to reduce class imbalance and improve reliability |
| Delivery Prediction | Add tip amount prediction; include location/zone data if available to improve accuracy |
| Demand Forecasting | Add location-level forecasting if zone data becomes available; test additional seasonality (monthly, holidays) |
| Overall Pipeline | Automate the flow from raw data to business report so insights can be refreshed on a schedule |